In [ ]:
# Generate Image

> Generate image and mask 

In [ ]:
#| default_exp noise_sharpen_gen

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import cv2
from pathlib import Path
import albumentations as A
from fastcore.all import *
from typing import List, Tuple, Union
import numpy as np
from tqdm.auto import tqdm
from cv_tools.core import *
from fastcore.all import *
from argparse import ArgumentParser
import shutil

In [ ]:
#| export
def noise_img_save(
    src_img_path:str,
    src_img_name:str,
    src_msk_path:str,
    src_msk_name:str,
    dst_msk_path:str,
    dst_img_path:str,
    src_img1_path:str=None,
    dst_img1_path:str=None,
    src_im1_name:str=None,
    var_list= [50,60, 70, 80],
    mean_list= [0,10,8, 12],
    sharpen:bool=True,
    blur:bool=True,
    lighting:bool=True,
    alpha_list:List[float]=[0.5, 0.7, 0.8, 0.9, 1.2, 1.3],
    
    ):
    "Add gaussian noise to image and save it locally"


    img = cv2.imread(
                     f'{src_img_path}/{src_img_name}',
                     cv2.IMREAD_GRAYSCALE)
        
    for i in var_list:
        augmentation = A.Compose([
            A.GaussNoise(var_limit=(10, i), mean=0, always_apply=True),
            ])
        aug_img = augmentation(image=img)['image']
        cv2.imwrite(
            f'{dst_img_path}/{Path(src_img_name).stem}_var_{i}.png',
            aug_img)
        if dst_msk_path is not None:
            shutil.copyfile(f'{src_msk_path}/{src_msk_name}', f'{dst_msk_path}/{Path(src_msk_name).stem}_var_{i}.png')
        if src_img1_path is not None:
            shutil.copyfile(f'{src_img1_path}/{src_im1_name}', f'{dst_img1_path}/{Path(src_im1_name).stem}_var_{i}.png')

    for i in mean_list:
        augmentation = A.Compose([
            A.GaussNoise(var_limit=(10, 50), mean=i, always_apply=True),
            ])
        aug_img = augmentation(image=img)['image']
        cv2.imwrite(
            f'{dst_img_path}/{Path(src_img_name).stem}_mean_{i}.png',
            aug_img)
        if dst_msk_path is not None: 
            shutil.copyfile(f'{src_msk_path}/{src_msk_name}', f'{dst_msk_path}/{Path(src_msk_name).stem}_mean_{i}.png')
        if src_img1_path is not None:
            shutil.copyfile(f'{src_img1_path}/{src_im1_name}', f'{dst_img1_path}/{Path(src_im1_name).stem}_mean_{i}.png')
    if blur:
        for i in [3, 5, 7, 9, 11] :

            augmentation = A.Compose([
                A.GaussianBlur(blur_limit=(3, i), p=1.0),  # You can adjust the blur_limit and probability
            ])
            aug_img = augmentation(image=img)['image']
            cv2.imwrite(
                f'{dst_img_path}/{Path(src_img_name).stem}_blur_kernel_{i}.png',
                aug_img)
            if dst_msk_path is not None:
                shutil.copyfile(f'{src_msk_path}/{src_msk_name}', f'{dst_msk_path}/{Path(src_msk_name).stem}_blur_kernel_{i}.png')
            if src_img1_path is not None:
                shutil.copyfile(f'{src_img1_path}/{src_im1_name}', f'{dst_img1_path}/{Path(src_im1_name).stem}_blur_kernel_{i}.png') 
    if sharpen:
        kernel = np.array([[-1, -1, -1],
                   [-1,  9, -1],
                   [-1, -1, -1]])
        sharpen_cv = cv2. filter2D(img, -1, kernel)
        cv2.imwrite(
            f'{dst_img_path}/{Path(src_img_name).stem}_sharpen_image.png',
            sharpen_cv)
        if dst_msk_path is not None:
            shutil.copyfile(
                f'{src_msk_path}/{src_msk_name}', 
                f'{dst_msk_path}/{Path(src_msk_name).stem}_sharpen_image.png')
        if src_img1_path is not None:
            shutil.copyfile(
                f'{src_img1_path}/{src_im1_name}', 
                f'{dst_img1_path}/{Path(src_im1_name).stem}_sharpen_image.png')
    if lighting:
        for i in alpha_list:
            img = read_img(f'{src_img_path}/{src_img_name}')
            lit_img = adjust_brightness(img, alpha=i)
            new_name = f'{Path(src_img_name).stem}_alpha_{i}_image.png'
            im1_name = new_name.replace('image2', 'image1')
            cv2.imwrite(
                f'{dst_img_path}/{new_name}',
                lit_img)
            if dst_msk_path is not None:
                shutil.copyfile(
                    f'{src_msk_path}/{src_msk_name}', 
                    f'{dst_msk_path}/{new_name}')
            if src_img1_path is not None:
                shutil.copyfile(
                f'{src_img1_path}/{src_im1_name}', 
                f'{dst_img1_path}/{Path(src_im1_name).stem}_alpha_{i}_image.png')


In [ ]:
#| export
def save_noised_img(
    src_img_path:str,
    dst_img_path:str,
    src_msk_path:str,
    dst_msk_path:str,
    src_img1_path:str=None,
    dst_img1_path:str=None,
    sharpen:bool=True,
    blur:bool=True,
    lighting:bool=True,
                   ):
    dst_img_path = Path(dst_img_path)
    dst_img_path.mkdir(exist_ok=True, parents=True)
    src_img_path = Path(src_img_path)

    if dst_img1_path is not None:
        dst_img1_path = Path(dst_img1_path)
        dst_img1_path.mkdir(exist_ok=True, parents=True)

    if dst_msk_path is not None:
        dst_msk_path = Path(dst_msk_path)
        dst_msk_path.mkdir(exist_ok=True, parents=True)


        src_msk_path = Path(src_msk_path)

    for i in tqdm(src_img_path.ls()):
        name = Path(i).name
        img1_name = name.replace('image2', 'image1')
        noise_img_save(
            src_img_path=src_img_path,
            src_msk_name=name,
            src_img_name=name,
            src_msk_path=src_msk_path,
            dst_msk_path=dst_msk_path,
            dst_img_path=dst_img_path,
            src_im1_name=img1_name,
            src_img1_path=src_img1_path,
            dst_img1_path=dst_img1_path,
            blur=blur,
            sharpen=sharpen,
            lighting=lighting

        )


In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser()
    parser.add_argument(
                       '-sip',
                       '--src_img_path',
                       type=str,
                       help='source image path, which will be used to generate noised image'
    )
    parser.add_argument(
                       '-dip',
                       '--dst_img_path',
                       type=str,
                       help='destination image path, where generated image will be saved'
    )
    parser.add_argument(
                       '-sip1',
                       '--src_img1_path',
                       default=None,
                       help='source image1(not image2) path, which will be used to generate noised image'
    )
    parser.add_argument(
                       '-dip1',
                       '--dst_img1_path',
                       default=None,
                       type=str,
                       help='destination image1 path, where generated image will be saved'
    )
    parser.add_argument(
        '--sharpen',
        action='store_true',
        help='whether to add sharpeing or not'
    )
    parser.add_argument(
        '--lighting',
        action='store_true',
        help='whether to add blurring or not'
    )
    parser.add_argument(
        '--blur',
        action='store_true',
        help='whether to add blurring or not'
    )
    parser.add_argument(
                       '-smp',
                       '--src_msk_path',
                       default=None,
                       help='Source mask path'
    )
    parser.add_argument(
                       '-dmp',
                       '--dst_msk_path',
                       default=None,
                       help='Destination mask path, where generated masks will be saved'
    )
    return parser.parse_args()

In [ ]:
#| export
def main_():
    args = parse_args_()

    save_noised_img(

        src_img_path=args.src_img_path,
        src_msk_path=args.src_msk_path,
        dst_img_path=args.dst_img_path,
        dst_msk_path=args.dst_msk_path,
        src_img1_path=args.src_img1_path,
        dst_img1_path=args.dst_img1_path,
        blur=args.blur,
        sharpen=args.sharpen,
        lighting=args.lighting


    )

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
Path.cwd()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('13_noise_lighting.ipynb')